# Потоки в Python

Потоки легковеснее процессов, имеют меньше накладных расходов, могут работать с любыми типами данных и не заставляют переживать, сможете ли вы сериализовать данные для передачи потоку.

Потоки можно создать с помощью модуля `threading`. Работа с потоками похожа на работу с процессами.

In [ ]:
import time

# Вместо multiprocessing используем threading
from threading import Thread


def printer(name):
    time.sleep(3)
    print('привет', name)


# Интерфейс похож на работу с процессами
t = Thread(target=printer, args=('Алиса',))
t.start()
print('Пока выполняется процесс, съешьте ещё этих мягких французских булок да выпейте чаю ☕️ ')
# Дождёмся выполнения операции в потоке потока
t.join()

Пока выполняется процесс, съешьте ещё этих мягких французских булок да выпейте чаю ☕️ 
привет Алиса


GIL (Global Interpreter Lock) позволяет работать только одному потоку внутри процесса, что снижает производительность многопоточных программ.

Управление памятью в Python — не потокобезопасное, поэтому нужна глобальная блокировка для корректного управления. Во время работы Python подсчитывает количество ссылок на объекты для корректного управления памятью. Это означает, что созданные в Python объекты имеют переменную подсчёта ссылок — refcount. В ней хранится количество всех ссылок на этот объект. Как только эта переменная становится равной нулю, память, выделенная под этот объект, освобождается. 

In [ ]:
import sys

# Переменная 'a' первая ссылается на объект object()
a = object()
# Переменная 'b' вторая ссылается на объект object()
b = a
# Так как вы передаёте переменную 'a', которая ссылается на объект, аргумент функции тоже будет ссылаться на этот объект
# В итоге на объект будет указывать три ссылки
print(sys.getrefcount(a))

3


В многопоточном приложении одновременно несколько потоков могут изменять значение счётчика ссылок. Это может привести к тому, что память очистится неправильно и удалится тот объект, на который ещё существует ссылка.

Счётчик ссылок можно защитить, добавив блокировки на все структуры данных, которые распространяются по нескольким потокам. Но добавление блокировки к нескольким объектам может привести к взаимоблокировки или deadlocks, которая получается, если блокировка есть более чем на одном объекте, а также к снижению производительности из-за многократной установки блокировки

Если программа в основном занята операциями ввода-вывода — запросами к сервисам, запросами к БД, чтением файлов — то использование потоков позволит распараллелить программу. GIL не будет мешать, если поток будет выполнять операцию ввода/вывода, так как в этот момент он точно не будет менять счётчик ссылок. GIL просто передаст управление другому потоку, пока первый поток ожидает завершения операции ввода/вывода

## Состояние гонки

Состояние гонки — это ошибка проектирования многопоточного приложения, при которой его работа зависит от того, в каком порядке выполняются части кода. Это может произойти при работе с общими объектами между потоками, несмотря на то что GIL позволяет работать только одному потоку в один момент времени.

In [4]:
import time
import os

from threading import Thread

# Глобальная переменная, которая будет изменяться из нескольких потоков
counter = 0

def increment():
    global counter
    for i in range(2000000):
        counter += int(1)

# Создание и запуск потоков
t = Thread(target=increment)
t.start()

t2 = Thread(target=increment)
t2.start()

# Дожидаемся завершения работы потоков
t.join()
t2.join()
print(counter)

2290568


Ожидаемый результат увеличения счётчика — 4000000. По факту результаты каждый раз разные. Это происходит потому, что изменения переменной counter происходят не атомарно. Сначала в буфер копируется значение переменной counterю. Допустим, текущее значение counter — 100. В обоих потоках в переменную buf запишется значение 100. Оба потока увеличивают значение буфера и получают 101. Переменная buf у каждого потока своя. Оба потока сохраняют переменную buf в counter. В итоге counter будет равен 101, а не 102.

In [5]:
import time
import os

from threading import Thread, Lock

counter = 0

# Объект блокировки
lock = Lock()

def increment():
    global counter
    for i in range(2000000):
        # С помощью контекстного менеджера захватываем блокировку и отпускаем, как только выходим из него
        with lock:
            counter += int(1)


t = Thread(target=increment)
t.start()

t2 = Thread(target=increment)
t2.start()

t.join()
t2.join()
print(counter)

4000000
